# 03 — Geospatial Visualization

Interactive mapping of geothermal prospects using Folium and Matplotlib.

**Objectives:**
- Create interactive prospect maps with clickable markers
- Generate heatmap overlays showing gradient intensity
- Compare wells visually on a geographic context

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.well_analysis import TemperatureGradient
from src.geospatial_viz import create_prospect_map, create_heatmap, plot_gradient_comparison

%matplotlib inline

## 3.1 Prepare Data

First, compute gradients and merge with location data.

In [ ]:
# Load and process data
data = pd.read_csv('../data/sample_wells.csv').dropna(subset=['temperature_c'])

# Compute gradients
tg = TemperatureGradient(data)
results = tg.compute_all()

# Merge gradient results with well locations
locations = data.groupby('well_id').agg({'latitude': 'first', 'longitude': 'first'}).reset_index()
map_data = results.merge(locations, on='well_id')

# Add anomaly classification
anomalies = tg.detect_anomalies(threshold_percentile=75)
map_data = map_data.merge(anomalies[['well_id', 'classification']], on='well_id', how='left')

print(f'Map data prepared: {len(map_data)} wells with coordinates and gradients')

## 3.2 Interactive Prospect Map

Clickable markers coloured by temperature gradient intensity.

In [ ]:
# Create interactive map
prospect_map = create_prospect_map(
    map_data,
    color_by='gradient_C_per_km',
    title='BCS Geothermal Prospects — Temperature Gradient'
)

# Save as HTML
prospect_map.save('../images/prospect_map.html')
print('Interactive map saved to images/prospect_map.html')

# Display in notebook
prospect_map

## 3.3 Gradient Heatmap

Spatial distribution of thermal anomalies.

In [ ]:
# Create heatmap
heatmap = create_heatmap(map_data, value_col='gradient_C_per_km', radius=30)
heatmap.save('../images/gradient_heatmap.html')
print('Heatmap saved to images/gradient_heatmap.html')
heatmap

## 3.4 Static Gradient Comparison Chart

In [ ]:
# Bar chart comparison
fig = plot_gradient_comparison(map_data, top_n=15, save_path='../images/03_gradient_comparison.png')
plt.show()

## 3.5 Scatter Map: Gradient vs Depth

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    map_data['longitude'], map_data['latitude'],
    c=map_data['gradient_C_per_km'], cmap='YlOrRd',
    s=map_data['max_depth_m'] / 10,  # Size proportional to depth
    alpha=0.8, edgecolors='black', linewidth=0.5
)

for _, row in map_data.iterrows():
    ax.annotate(row['well_id'], (row['longitude'], row['latitude']),
                fontsize=7, ha='center', va='bottom')

cbar = plt.colorbar(scatter, ax=ax, label='Gradient (°C/km)')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Geothermal Prospects — Gradient & Depth', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../images/03_scatter_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Next:** [04 — Monte Carlo Resource Assessment](04_monte_carlo_resource_assessment.ipynb)